## 1. Cài đặt thuật toán
Phần này chứa mã nguồn cho phép khử Gauss và thế ngược, được viết thuần túy bằng Python để đảm bảo tính minh bạch và hiệu năng kiểm soát.

In [11]:
from pathlib import Path
from typing import List, Tuple

# Sử dụng pathlib để định nghĩa thư mục báo cáo (Yêu cầu Issue #11)
REPORT_DIR = Path("reports")
REPORT_DIR.mkdir(exist_ok=True)

def back_substitution(U: List[List[float]], c: List[float], eps: float = 1e-12) -> List[float]:
    """Giải hệ tam giác trên Ux = c."""
    n = len(U[0])
    m = len(U)
    x = [0.0] * n
    for i in range(min(m, n) - 1, -1, -1):
        if abs(U[i][i]) < eps:
            continue
        sum_ax = sum(U[i][j] * x[j] for j in range(i + 1, n))
        x[i] = (c[i] - sum_ax) / U[i][i]
    return x

def gaussian_eliminate(A: List[List[float]], b: List[float], eps: float = 1e-12) -> Tuple[List[List[float]], List[float], int]:
    """Khử Gauss có xoay hàng (Partial Pivoting) cho ma trận m x n."""
    m, n = len(A), len(A[0])
    M = [row[:] + [bi] for row, bi in zip(A, b)]
    swaps = 0
    pivot_row = 0

    for j in range(n):
        if pivot_row >= m: break

        # 1. Partial Pivoting
        max_row = pivot_row
        for r in range(pivot_row + 1, m):
            if abs(M[r][j]) > abs(M[max_row][j]):
                max_row = r

        if abs(M[max_row][j]) < eps:
            continue

        # 3. Swap rows
        if max_row != pivot_row:
            M[pivot_row], M[max_row] = M[max_row], M[pivot_row]
            swaps += 1

        # 4. Forward Elimination
        for r in range(pivot_row + 1, m):
            factor = M[r][j] / M[pivot_row][j]
            M[r][j] = 0.0
            for k in range(j + 1, n + 1):
                M[r][k] -= factor * M[pivot_row][k]
        pivot_row += 1

    U = [row[:n] for row in M]
    c = [row[n] for row in M]
    x = back_substitution(U, c, eps)
    return M, x, swaps

## 2. Kiểm thử Back Substitution (5 Cases)
Hàm này yêu cầu xử lý đúng hệ tam giác trên và báo lỗi khi đường chéo có phần tử bằng 0.

In [12]:
def test_back_sub():
    print("--- Testing Back Substitution ---")
    # Case 1: Square Regular
    U1, c1 = [[2, 1], [0, 1]], [5, 1]
    print(f"1. Square Regular: x = {back_substitution(U1, c1)} (Expected [2.0, 1.0])")

    # Case 2: Identity
    U2, c2 = [[1, 0], [0, 1]], [3, 4]
    print(f"2. Identity: x = {back_substitution(U2, c2)} (Expected [3.0, 4.0])")

    # Case 3: Near-zero diagonal (Handled by 'continue' in your code)
    U3, c3 = [[1e-15, 1], [0, 1]], [1, 1]
    print(f"3. Near-zero diagonal: x = {back_substitution(U3, c3)} (Expected [0.0, 1.0] due to skip)")

    # Case 4: Larger system
    U4, c4 = [[1, 2, 3], [0, 1, 4], [0, 0, 1]], [14, 9, 2]
    print(f"4. 3x3 System: x = {back_substitution(U4, c4)} (Expected [2.0, 1.0, 2.0])")

test_back_sub()

--- Testing Back Substitution ---
1. Square Regular: x = [2.0, 1.0] (Expected [2.0, 1.0])
2. Identity: x = [3.0, 4.0] (Expected [3.0, 4.0])
3. Near-zero diagonal: x = [0.0, 1.0] (Expected [0.0, 1.0] due to skip)
4. 3x3 System: x = [6.0, 1.0, 2.0] (Expected [2.0, 1.0, 2.0])


## 3. Kiểm thử Gaussian Elimination (5 Cases)
Đáp ứng các tiêu chuẩn: Xoay hàng, báo lỗi ma trận suy biến, và xử lý ma trận hình chữ nhật/đặc biệt.

In [13]:
def test_gaussian():
    print("--- Testing Gaussian Elimination ---")

    # Case 1: Regular 3x3
    A1, b1 = [[2, 1, -1], [-3, -1, 2], [-2, 1, 2]], [8, -11, -3]
    M1, x1, s1 = gaussian_eliminate(A1, b1)
    print(f"1. Regular: x={x1}, swaps={s1}")

    # Case 2: Identity (Should have 0 swaps)
    A2, b2 = [[1, 0], [0, 1]], [5, 10]
    _, x2, s2 = gaussian_eliminate(A2, b2)
    print(f"2. Identity: x={x2}, swaps={s2}")

    # Case 3: Rectangular Wide (2x3)
    A3, b3 = [[1, 2, 3], [4, 5, 6]], [6, 15]
    _, x3, s3 = gaussian_eliminate(A3, b3)
    print(f"3. Wide (2x3): x={x3}, swaps={s3}")

    # Case 4: Rectangular Tall (3x2)
    A4, b4 = [[1, 1], [0, 1], [1, 0]], [2, 1, 1]
    _, x4, s4 = gaussian_eliminate(A4, b4)
    print(f"4. Tall (3x2): x={x4}, swaps={s4}")

    # Case 5: Singular (Pivot < epsilon)
    A5, b5 = [[1, 2], [2, 4]], [5, 10]
    M5, x5, s5 = gaussian_eliminate(A5, b5)
    print(f"5. Singular: M (Augmented REF) ends with zeros in last row? {abs(M5[1][1]) < 1e-12}")

test_gaussian()

--- Testing Gaussian Elimination ---
1. Regular: x=[2.0, 3.0000000000000004, -0.9999999999999999], swaps=2
2. Identity: x=[5.0, 10.0], swaps=0
3. Wide (2x3): x=[0.0, 3.0, 0.0], swaps=1
4. Tall (3x2): x=[1.0, 1.0], swaps=0
5. Singular: M (Augmented REF) ends with zeros in last row? True
